# Notebook 03: Feature Engineering

This notebook creates lag features, rolling windows, wind vectors, cyclical time encodings, and applies PCA to co-pollutants.

In [2]:
import pandas as pd
import numpy as np
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

## 1. Load Processed Data

In [3]:
df = pd.read_csv('../data/processed/02_processed.csv', parse_dates=['datetime'], index_col='datetime')
print(f"Loaded {len(df)} rows.")

Loaded 105189 rows.


## 2. Lag Features & Rolling Windows
We need to create lags ($t-1, t-3, t-24$) for all numerical columns to avoid data leakage.

In [4]:
def create_lags(df, cols, lags):
    df_lagged = df.copy()
    for col in cols:
        for lag in lags:
            df_lagged[f'{col}_lag_{lag}'] = df_lagged.groupby('station')[col].shift(lag)
        # Rolling 24h mean
        df_lagged[f'{col}_roll_24'] = df_lagged.groupby('station')[col].transform(lambda x: x.rolling(24, min_periods=1).mean())
    return df_lagged

lag_cols = ['PM2.5', 'PM10', 'SO2', 'NO2', 'CO', 'O3', 'TEMP', 'PRES', 'DEWP', 'RAIN', 'WSPM']
df_feat = create_lags(df, lag_cols, [1, 3, 24])

## 3. Wind Vectors ($u, v$)
Convert wind direction (wd) and speed (WSPM) into orthogonal vectors.

In [5]:
# Define angles for compass directions
wd_angles = {
    'N': 0, 'NNE': 22.5, 'NE': 45, 'ENE': 67.5, 'E': 90, 'ESE': 112.5, 'SE': 135, 'SSE': 157.5,
    'S': 180, 'SSW': 202.5, 'SW': 225, 'WSW': 247.5, 'W': 270, 'WNW': 292.5, 'NW': 315, 'NNW': 337.5
}
df_feat['wd_angle'] = df_feat['wd'].map(wd_angles)

# Convert to radians
wd_rad = np.deg2rad(df_feat['wd_angle'])

# Calculate u and v (Note: Meteorological wind is where it comes FROM, so we use negative sin/cos for flow direction, but standard sin/cos is also fine as a feature)
df_feat['u_wind'] = -df_feat['WSPM'] * np.sin(wd_rad)
df_feat['v_wind'] = -df_feat['WSPM'] * np.cos(wd_rad)

# Drop original wind cols
df_feat.drop(['wd', 'wd_angle', 'WSPM'], axis=1, inplace=True)

## 4. Cyclical Encoding

In [6]:
df_feat['hour_sin'] = np.sin(2 * np.pi * df_feat['hour']/23.0)
df_feat['hour_cos'] = np.cos(2 * np.pi * df_feat['hour']/23.0)
df_feat['month_sin'] = np.sin(2 * np.pi * df_feat['month']/12.0)
df_feat['month_cos'] = np.cos(2 * np.pi * df_feat['month']/12.0)
# Drop original temporal vars except year (useful for split)
df_feat.drop(['hour', 'month', 'day'], axis=1, inplace=True)

## 5. PCA on Co-pollutants
We apply PCA to the *lagged* co-pollutants to solve multicollinearity.

In [7]:
# We must use only lagged versions (t-1) for PCA to avoid leakage
pca_cols = ['PM10_lag_1', 'SO2_lag_1', 'NO2_lag_1', 'CO_lag_1', 'O3_lag_1']

# Drop rows where lags are NaN (first 24 hours of each station)
df_feat.dropna(subset=pca_cols, inplace=True)

scaler = StandardScaler()
scaled_pollutants = scaler.fit_transform(df_feat[pca_cols])

pca = PCA(n_components=2) # Keep top 2 components
pca_features = pca.fit_transform(scaled_pollutants)

print(f"Explained variance ratio: {pca.explained_variance_ratio_}")

df_feat['pollutant_pca_1'] = pca_features[:, 0]
df_feat['pollutant_pca_2'] = pca_features[:, 1]

# Final cleanup of NaNs created by 24h lag
df_feat.dropna(inplace=True)
print(f"Final feature set shape: {df_feat.shape}")

# Save
df_feat.to_csv('../data/processed/03_features.csv')

Explained variance ratio: [0.57977716 0.19020961]
Final feature set shape: (105117, 65)
